Student B — Classical Models
Project 16: Credit Card Fraud Detection
Implements: Logistic Regression, Random Forest, XGBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
    average_precision_score, confusion_matrix, RocCurveDisplay,
    PrecisionRecallDisplay)
import xgboost as xgb
import joblib, os, time



In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
RESULTS_DIR = r"D:\Queens\DL-CISC 867-Deep Learning\DL Project\credit-card-fraud-detection\results"
MODELS_DIR  = r"D:\Queens\DL-CISC 867-Deep Learning\DL Project\credit-card-fraud-detection\models"
SEED        = 42
os.makedirs(MODELS_DIR, exist_ok=True)


In [ ]:
# ─── LOAD DATA (produced by Student A) ───────────────────────────────────────
X_train = np.load(f"{RESULTS_DIR}/X_train_sm.npy")   # SMOTE-balanced
y_train = np.load(f"{RESULTS_DIR}/y_train_sm.npy")
X_val   = np.load(f"{RESULTS_DIR}/X_val.npy")
y_val   = np.load(f"{RESULTS_DIR}/y_val.npy")
X_test  = np.load(f"{RESULTS_DIR}/X_test.npy")
y_test  = np.load(f"{RESULTS_DIR}/y_test.npy")

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (398038, 30), Val: (28481, 30), Test: (56962, 30)


In [ ]:
# ─── HELPER ──────────────────────────────────────────────────────────────────
def evaluate_model(name, model, X_te, y_te):
    t0 = time.time()
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = model.predict(X_te)
    elapsed = time.time() - t0

    roc  = roc_auc_score(y_te, y_prob)
    pr   = average_precision_score(y_te, y_prob)
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"ROC-AUC:  {roc:.4f}  |  PR-AUC: {pr:.4f}")
    print(f"Inference time: {elapsed:.3f}s")
    print(classification_report(y_te, y_pred, target_names=['Legit', 'Fraud']))
    return {"name": name, "roc_auc": roc, "pr_auc": pr, "time_s": elapsed,
            "y_prob": y_prob, "y_pred": y_pred}

In [ ]:
# ─── 1. LOGISTIC REGRESSION ──────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced', C=0.1)
lr.fit(X_train, y_train)
joblib.dump(lr, f"{MODELS_DIR}/logistic_regression.pkl")
res_lr = evaluate_model("Logistic Regression", lr, X_test, y_test)



Model: Logistic Regression
ROC-AUC:  0.9731  |  PR-AUC: 0.7258
Inference time: 0.003s
              precision    recall  f1-score   support

       Legit       1.00      0.97      0.99     56864
       Fraud       0.06      0.91      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.94      0.55     56962
weighted avg       1.00      0.97      0.99     56962



In [ ]:
# ─── 2. RANDOM FOREST ────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=4,
                             class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
joblib.dump(rf, f"{MODELS_DIR}/random_forest.pkl")
res_rf = evaluate_model("Random Forest", rf, X_test, y_test)

# Feature importance
feat_imp = pd.Series(rf.feature_importances_,
    index=[f"V{i}" for i in range(1, 29)] + ["Amount_scaled", "Time_scaled"])
feat_imp.nlargest(15).sort_values().plot(kind='barh', color='steelblue')
plt.title("Random Forest — Top 15 Feature Importances")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/rf_feature_importance.png", dpi=150)
plt.close()



Model: Random Forest
ROC-AUC:  0.9860  |  PR-AUC: 0.8165
Inference time: 0.155s
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.58      0.85      0.69        98

    accuracy                           1.00     56962
   macro avg       0.79      0.92      0.84     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
# ─── 3. XGBOOST ──────────────────────────────────────────────────────────────
scale_pos = int((y_train == 0).sum() / (y_train == 1).sum())
xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos,
    random_state=SEED, use_label_encoder=False, eval_metric='aucpr', n_jobs=-1)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=50)
xgb_model.save_model(f"{MODELS_DIR}/xgboost.json")
res_xgb = evaluate_model("XGBoost", xgb_model, X_test, y_test)


[0]	validation_0-aucpr:0.46514


c:\Users\DPQUAI250103\AppData\Local\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:05:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-aucpr:0.66530
[100]	validation_0-aucpr:0.72792
[150]	validation_0-aucpr:0.76836
[200]	validation_0-aucpr:0.77989
[250]	validation_0-aucpr:0.78317
[299]	validation_0-aucpr:0.78504

Model: XGBoost
ROC-AUC:  0.9787  |  PR-AUC: 0.8700
Inference time: 0.054s
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.57      0.88      0.69        98

    accuracy                           1.00     56962
   macro avg       0.79      0.94      0.85     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
# ─── 4. COMPARISON PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results = [res_lr, res_rf, res_xgb]
names   = [r["name"] for r in results]

# ROC curves
for r in results:
    RocCurveDisplay.from_predictions(y_test, r["y_prob"],
        name=r["name"], ax=axes[0])
axes[0].set_title("ROC Curves — Classical Models")

# PR curves
for r in results:
    PrecisionRecallDisplay.from_predictions(y_test, r["y_prob"],
        name=r["name"], ax=axes[1])
axes[1].set_title("Precision-Recall Curves — Classical Models")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/classical_roc_pr_curves.png", dpi=150)
plt.close()


In [ ]:
# ─── 5. SUMMARY TABLE ────────────────────────────────────────────────────────
summary = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_prob','y_pred']}
                         for r in results])
summary.to_csv(f"{RESULTS_DIR}/classical_model_summary.csv", index=False)
print(f"\n{summary.to_string()}")
print("\nClassical models complete. Results saved.")



                  name   roc_auc    pr_auc    time_s
0  Logistic Regression  0.973080  0.725819  0.003284
1        Random Forest  0.986043  0.816518  0.154747
2              XGBoost  0.978725  0.869984  0.054142

Classical models complete. Results saved.
